# Assessment 2 - Applied NLP Project
## AG News Robustness Benchmark

This notebook presents an applied NLP project for Assessment 2.

The project compares how three text classifiers behave on clean AG News text and on deliberately corrupted text.

The three models are:
- word n-gram TF-IDF + Linear SVM
- character n-gram TF-IDF + Linear SVM
- a pretrained transformer classifier fine-tuned on AG News

The character model is included because it is usually more resistant to typos and lexical noise, which makes the robustness story more interesting.

In [3]:
"""AG News robustness benchmark.

This script builds a small applied NLP project around the AG News dataset.
It covers:
- dataset understanding and EDA,
- preprocessing,
- three text classification models,
- robustness testing under noisy text perturbations,
- and saved outputs for the final report.

The default benchmark compares:
- a word n-gram TF-IDF + Linear SVM model,
- a character n-gram TF-IDF + Linear SVM model.
- a pretrained transformer classifier fine-tuned on AG News.

The character model is included on purpose because it is often more robust to
typos and lexical corruption, which gives us an interesting robustness story.
"""

from __future__ import annotations

import argparse
import json
import math
import random
import re
from pathlib import Path

import matplotlib

matplotlib.use("Agg")

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from datasets import load_dataset
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC
from transformers import AutoModelForSequenceClassification, AutoTokenizer


LABEL_NAMES = {
    0: "World",
    1: "Sports",
    2: "Business",
    3: "Sci/Tech",
}
LABEL_ORDER = [LABEL_NAMES[idx] for idx in sorted(LABEL_NAMES)]

ATTACK_PLAN = [
    ("typo", 0.05),
    ("typo", 0.10),
    ("typo", 0.20),
    ("dropout", 0.05),
    ("dropout", 0.10),
    ("dropout", 0.20),
    ("vowel_mask", 0.05),
    ("vowel_mask", 0.10),
    ("vowel_mask", 0.20),
]

TRANSFORMER_MODEL_NAME = "textattack/distilbert-base-uncased-ag-news"


def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser(
        description="Build an AG News robustness benchmark project."
    )
    parser.add_argument(
        "--output-dir",
        type=Path,
        default=Path("outputs"),
        help="Directory where plots and CSV/JSON outputs will be written.",
    )
    parser.add_argument(
        "--train-sample",
        type=int,
        default=None,
        help="Optional train sample size for faster local runs.",
    )
    parser.add_argument(
        "--test-sample",
        type=int,
        default=None,
        help="Optional test sample size for faster local runs.",
    )
    parser.add_argument(
        "--seed",
        type=int,
        default=42,
        help="Random seed used for sampling and perturbations.",
    )
    return parser.parse_args()


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)


def normalize_text(text: str) -> str:
    text = str(text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def sample_frame(df: pd.DataFrame, sample_size: int | None, seed: int) -> pd.DataFrame:
    if sample_size is None or sample_size >= len(df):
        return df.reset_index(drop=True).copy()
    return df.sample(n=sample_size, random_state=seed).reset_index(drop=True).copy()


def load_ag_news(train_sample: int | None, test_sample: int | None, seed: int) -> tuple[pd.DataFrame, pd.DataFrame]:
    dataset = load_dataset("ag_news")

    train_df = dataset["train"].to_pandas()
    test_df = dataset["test"].to_pandas()

    train_df = sample_frame(train_df, train_sample, seed)
    test_df = sample_frame(test_df, test_sample, seed + 1)

    for df, split_name in ((train_df, "train"), (test_df, "test")):
        df["text"] = df["text"].map(normalize_text)
        df["label_name"] = df["label"].map(LABEL_NAMES)
        df["split"] = split_name
        df["char_count"] = df["text"].str.len()
        df["word_count"] = df["text"].str.split().str.len()

    return train_df, test_df


def make_output_dir(output_dir: Path) -> None:
    output_dir.mkdir(parents=True, exist_ok=True)


def save_json(payload: dict, path: Path) -> None:
    path.write_text(json.dumps(payload, indent=2, sort_keys=True), encoding="utf-8")


def compute_text_summary(df: pd.DataFrame) -> dict:
    summary = {
        "rows": int(len(df)),
        "class_counts": {k: int(v) for k, v in df["label_name"].value_counts().reindex(LABEL_ORDER).fillna(0).items()},
        "char_count": {
            "mean": float(df["char_count"].mean()),
            "median": float(df["char_count"].median()),
            "p90": float(df["char_count"].quantile(0.90)),
            "min": int(df["char_count"].min()),
            "max": int(df["char_count"].max()),
        },
        "word_count": {
            "mean": float(df["word_count"].mean()),
            "median": float(df["word_count"].median()),
            "p90": float(df["word_count"].quantile(0.90)),
            "min": int(df["word_count"].min()),
            "max": int(df["word_count"].max()),
        },
    }
    return summary


def save_eda_plots(df: pd.DataFrame, output_dir: Path) -> None:
    sns.set_theme(style="whitegrid", context="talk")

    class_counts = df["label_name"].value_counts().reindex(LABEL_ORDER)
    class_df = class_counts.reset_index()
    class_df.columns = ["label_name", "count"]

    fig, ax = plt.subplots(figsize=(11, 6))
    sns.barplot(data=class_df, x="label_name", y="count", ax=ax, palette="Blues_r")
    ax.set_title("AG News class distribution")
    ax.set_xlabel("Label")
    ax.set_ylabel("Count")
    ax.bar_label(ax.containers[0], fmt="%d", padding=3, fontsize=11)
    fig.tight_layout()
    fig.savefig(output_dir / "eda_class_distribution.png", dpi=200)
    plt.close(fig)

    fig, axes = plt.subplots(1, 2, figsize=(15, 6))
    sns.boxplot(
        data=df,
        x="label_name",
        y="word_count",
        order=LABEL_ORDER,
        ax=axes[0],
        palette="Set2",
        showfliers=False,
    )
    axes[0].set_title("Word count by class")
    axes[0].set_xlabel("Label")
    axes[0].set_ylabel("Words per text")

    sns.histplot(
        data=df.sample(n=min(len(df), 20000), random_state=42),
        x="word_count",
        bins=40,
        ax=axes[1],
        color="#2c7fb8",
    )
    axes[1].set_title("Overall text length distribution")
    axes[1].set_xlabel("Words per text")
    axes[1].set_ylabel("Count")

    fig.tight_layout()
    fig.savefig(output_dir / "eda_text_length.png", dpi=200)
    plt.close(fig)


def write_sample_examples(df: pd.DataFrame, output_dir: Path, n_per_class: int = 3) -> None:
    examples = (
        df.groupby("label_name", group_keys=False)
        .apply(lambda group: group.sample(n=min(n_per_class, len(group)), random_state=42))
        .reset_index(drop=True)
    )
    examples[["label_name", "text"]].to_csv(output_dir / "sample_examples.csv", index=False)


def build_word_model() -> Pipeline:
    return Pipeline(
        steps=[
            (
                "tfidf",
                TfidfVectorizer(
                    lowercase=True,
                    stop_words="english",
                    ngram_range=(1, 2),
                    min_df=2,
                    max_df=0.95,
                    max_features=60000,
                    sublinear_tf=True,
                ),
            ),
            (
                "clf",
                LinearSVC(C=1.5),
            ),
        ]
    )


def build_char_model() -> Pipeline:
    return Pipeline(
        steps=[
            (
                "tfidf",
                TfidfVectorizer(
                    analyzer="char_wb",
                    ngram_range=(3, 5),
                    min_df=2,
                    max_features=90000,
                    sublinear_tf=True,
                ),
            ),
            (
                "clf",
                LinearSVC(C=1.0),
            ),
        ]
    )


class PretrainedTransformerClassifier:
    """Lightweight wrapper around a pretrained text classifier."""

    def __init__(
        self,
        model_name: str = TRANSFORMER_MODEL_NAME,
        batch_size: int = 32,
        max_length: int = 128,
    ) -> None:
        self.model_name = model_name
        self.batch_size = batch_size
        self.max_length = max_length
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForSequenceClassification.from_pretrained(model_name)
        self.model.to(self.device)
        self.model.eval()

    def fit(self, X: pd.Series | list[str], y: pd.Series | np.ndarray) -> "PretrainedTransformerClassifier":
        # This is a pretrained checkpoint, so there is no extra fitting step.
        return self

    def predict(self, texts: pd.Series | list[str]) -> np.ndarray:
        predictions: list[int] = []
        text_list = list(texts)

        for start in range(0, len(text_list), self.batch_size):
            batch_texts = text_list[start : start + self.batch_size]
            encoded = self.tokenizer(
                batch_texts,
                padding=True,
                truncation=True,
                max_length=self.max_length,
                return_tensors="pt",
            )
            encoded = {key: value.to(self.device) for key, value in encoded.items()}

            with torch.no_grad():
                logits = self.model(**encoded).logits

            batch_predictions = logits.argmax(dim=-1).detach().cpu().numpy().tolist()
            predictions.extend(batch_predictions)

        return np.asarray(predictions, dtype=int)


def build_transformer_model(
    model_name: str = TRANSFORMER_MODEL_NAME,
    batch_size: int = 32,
    max_length: int = 128,
) -> PretrainedTransformerClassifier:
    return PretrainedTransformerClassifier(
        model_name=model_name,
        batch_size=batch_size,
        max_length=max_length,
    )


def corrupt_word(word: str, rng: random.Random) -> str:
    if len(word) < 4:
        return word

    letters = list(word)
    operation = rng.choice(["swap", "delete", "duplicate"])

    if operation == "swap" and len(letters) > 3:
        idx = rng.randrange(1, len(letters) - 1)
        letters[idx], letters[idx + 1] = letters[idx + 1], letters[idx]
    elif operation == "delete":
        del letters[rng.randrange(1, len(letters) - 1)]
    else:
        idx = rng.randrange(1, len(letters) - 1)
        letters.insert(idx, letters[idx])

    return "".join(letters)


def apply_typo_noise(text: str, rng: random.Random, severity: float) -> str:
    def replace(match: re.Match[str]) -> str:
        word = match.group(0)
        if rng.random() > severity:
            return word
        return corrupt_word(word, rng)

    return re.sub(r"[A-Za-z]{4,}", replace, text)


def apply_word_dropout(text: str, rng: random.Random, severity: float) -> str:
    tokens = text.split()
    kept = [token for token in tokens if rng.random() > severity]
    if not kept and tokens:
        kept = [tokens[0]]
    return " ".join(kept)


def apply_vowel_mask(text: str, rng: random.Random, severity: float) -> str:
    def replace(match: re.Match[str]) -> str:
        word = match.group(0)
        if rng.random() > severity:
            return word
        masked = re.sub(r"[aeiouAEIOU]", "", word)
        return masked or word

    return re.sub(r"[A-Za-z]{4,}", replace, text)


def apply_attack(text: str, attack_name: str, severity: float, rng: random.Random) -> str:
    if attack_name == "typo":
        return apply_typo_noise(text, rng, severity)
    if attack_name == "dropout":
        return apply_word_dropout(text, rng, severity)
    if attack_name == "vowel_mask":
        return apply_vowel_mask(text, rng, severity)
    raise ValueError(f"Unknown attack: {attack_name}")


def perturb_corpus(texts: list[str], attack_name: str, severity: float, seed: int) -> list[str]:
    rng = random.Random(seed)
    return [apply_attack(text, attack_name, severity, rng) for text in texts]


def evaluate_predictions(y_true: np.ndarray, y_pred: np.ndarray) -> dict:
    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "macro_f1": float(f1_score(y_true, y_pred, average="macro")),
        "weighted_f1": float(f1_score(y_true, y_pred, average="weighted")),
    }


def save_confusion_matrix(
    y_true: np.ndarray,
    y_pred: np.ndarray,
    output_path: Path,
    title: str,
) -> None:
    cm = confusion_matrix(y_true, y_pred, labels=sorted(LABEL_NAMES))
    disp = ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=[LABEL_NAMES[idx] for idx in sorted(LABEL_NAMES)],
    )

    fig, ax = plt.subplots(figsize=(8, 7))
    disp.plot(ax=ax, cmap="Blues", colorbar=False, values_format="d")
    ax.set_title(title)
    fig.tight_layout()
    fig.savefig(output_path, dpi=200)
    plt.close(fig)


def save_metric_table(df: pd.DataFrame, path: Path) -> None:
    df.to_csv(path, index=False)


def build_model_summary(clean_df: pd.DataFrame, robustness_df: pd.DataFrame) -> pd.DataFrame:
    clean_lookup = clean_df.set_index("model")

    rows = []
    for model_name, group in robustness_df.groupby("model"):
        clean_metrics = clean_lookup.loc[model_name]
        rows.append(
            {
                "model": model_name,
                "clean_accuracy": float(clean_metrics["accuracy"]),
                "clean_macro_f1": float(clean_metrics["macro_f1"]),
                "avg_perturbed_accuracy": float(group["accuracy"].mean()),
                "avg_perturbed_macro_f1": float(group["macro_f1"].mean()),
                "worst_perturbed_macro_f1": float(group["macro_f1"].min()),
                "mean_macro_f1_drop": float(clean_metrics["macro_f1"] - group["macro_f1"].mean()),
                "worst_macro_f1_drop": float(clean_metrics["macro_f1"] - group["macro_f1"].min()),
            }
        )

    summary = pd.DataFrame(rows).sort_values(
        by=["avg_perturbed_macro_f1", "clean_macro_f1"], ascending=False
    )
    return summary


def save_robustness_curves(robustness_df: pd.DataFrame, output_dir: Path) -> None:
    sns.set_theme(style="whitegrid", context="talk")

    for attack_name in robustness_df["attack"].unique():
        subset = robustness_df[robustness_df["attack"] == attack_name].copy()
        subset["severity_pct"] = subset["severity"]

        fig, ax = plt.subplots(figsize=(10, 6))
        sns.lineplot(
            data=subset,
            x="severity_pct",
            y="macro_f1",
            hue="model",
            marker="o",
            ax=ax,
        )
        ax.xaxis.set_major_formatter(mticker.PercentFormatter(xmax=1.0))
        ax.set_title(f"Macro-F1 under {attack_name.replace('_', ' ')} noise")
        ax.set_xlabel("Noise severity")
        ax.set_ylabel("Macro-F1")
        ax.legend(title="Model")
        fig.tight_layout()
        fig.savefig(output_dir / f"robustness_{attack_name}.png", dpi=200)
        plt.close(fig)


def save_summary_text(summary_df: pd.DataFrame, output_dir: Path) -> None:
    best_row = summary_df.iloc[0]
    runner_up = summary_df.iloc[1] if len(summary_df) > 1 else None

    lines = [
        "AG News robustness benchmark summary",
        "===================================",
        "",
        f"Best model by average perturbed macro-F1: {best_row['model']}",
        f"  Clean accuracy: {best_row['clean_accuracy']:.4f}",
        f"  Clean macro-F1: {best_row['clean_macro_f1']:.4f}",
        f"  Avg perturbed macro-F1: {best_row['avg_perturbed_macro_f1']:.4f}",
        f"  Worst perturbed macro-F1: {best_row['worst_perturbed_macro_f1']:.4f}",
        "",
    ]

    if runner_up is not None:
        lines.extend(
            [
                f"Runner-up model: {runner_up['model']}",
                f"  Clean accuracy: {runner_up['clean_accuracy']:.4f}",
                f"  Clean macro-F1: {runner_up['clean_macro_f1']:.4f}",
                f"  Avg perturbed macro-F1: {runner_up['avg_perturbed_macro_f1']:.4f}",
                f"  Worst perturbed macro-F1: {runner_up['worst_perturbed_macro_f1']:.4f}",
                "",
            ]
        )

    lines.append("Interpretation notes:")
    lines.append("- Word n-grams usually score well on clean text.")
    lines.append("- Character n-grams should be more resistant to typos and vowel masking.")
    lines.append("- The transformer adds a stronger contextual baseline for comparison.")
    lines.append("- Word dropout is a strong stress test for all models.")

    (output_dir / "summary.txt").write_text("\n".join(lines), encoding="utf-8")



In [4]:
pd.set_option('display.max_colwidth', 120)

seed = 42
output_dir = Path('outputs')
make_output_dir(output_dir)
set_seed(seed)

seed

42

## 1. Problem Definition and Dataset Understanding

AG News is a 4-class topic classification dataset. The task is to assign each news snippet to one of the following labels:
- World
- Sports
- Business
- Sci/Tech

This is an NLP problem because the model must interpret free text and map it to a semantic category.
For the robustness part, we also test how well the model survives typos, missing words, and vowel masking.

One limitation of AG News is that the texts are short and fairly clean, so the task is more controlled than many real-world text classification problems. That makes it a good benchmark for comparing model robustness, but it is still a simplified view of how noisy production text can be.

In [5]:
train_df, test_df = load_ag_news(train_sample=None, test_sample=None, seed=seed)

eda_summary = {
    'train': compute_text_summary(train_df),
    'test': compute_text_summary(test_df),
}

eda_summary

README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/18.6M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/1.23M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/120000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/7600 [00:00<?, ? examples/s]

{'train': {'rows': 120000,
  'class_counts': {'World': 30000,
   'Sports': 30000,
   'Business': 30000,
   'Sci/Tech': 30000},
  'char_count': {'mean': 235.90625,
   'median': 231.0,
   'p90': 299.0,
   'min': 99,
   'max': 1012},
  'word_count': {'mean': 37.84745,
   'median': 37.0,
   'p90': 48.0,
   'min': 8,
   'max': 177}},
 'test': {'rows': 7600,
  'class_counts': {'World': 1900,
   'Sports': 1900,
   'Business': 1900,
   'Sci/Tech': 1900},
  'char_count': {'mean': 234.74276315789473,
   'median': 231.0,
   'p90': 298.0,
   'min': 100,
   'max': 880},
  'word_count': {'mean': 37.72236842105263,
   'median': 37.0,
   'p90': 48.0,
   'min': 11,
   'max': 137}}}

In [6]:
save_eda_plots(train_df, output_dir)
write_sample_examples(train_df, output_dir)

train_df.groupby('label_name')[['char_count', 'word_count']].mean().round(2)

/tmp/ipykernel_55/803510508.py:181: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(data=class_df, x="label_name", y="count", ax=ax, palette="Blues_r")
/tmp/ipykernel_55/803510508.py:191: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(
/tmp/ipykernel_55/803510508.py:223: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda group: group.sample(n=min(n_per_class, len(group)), random_state=42))


,char_count,word_count
label_name,,
Business,240.57,37.54
Sci/Tech,236.79,37.19
Sports,224.24,37.77
World,242.02,38.88


## 2. Preprocessing and Feature Extraction

The preprocessing is intentionally lightweight:
- trim and normalize whitespace,
- keep the text itself intact,
- let TF-IDF handle tokenization and weighting.

The two feature sets are:
- word n-grams for a strong semantic baseline,
- character n-grams for better typo tolerance.

In [7]:
models = {
    'word_ngram_svm': build_word_model(),
    'char_ngram_svm': build_char_model(),
    'transformer_distilbert': build_transformer_model(),
}

models

config.json:   0%|          | 0.00/680 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

{'word_ngram_svm': Pipeline(steps=[('tfidf',
                  TfidfVectorizer(max_df=0.95, max_features=60000, min_df=2,
                                  ngram_range=(1, 2), stop_words='english',
                                  sublinear_tf=True)),
                 ('clf', LinearSVC(C=1.5))]),
 'char_ngram_svm': Pipeline(steps=[('tfidf',
                  TfidfVectorizer(analyzer='char_wb', max_features=90000,
                                  min_df=2, ngram_range=(3, 5),
                                  sublinear_tf=True)),
                 ('clf', LinearSVC())]),
 'transformer_distilbert': <__main__.PretrainedTransformerClassifier at 0x7c1a86b124e0>}

## 3. Model Development and Clean Evaluation

We evaluate the two classical baselines and the pretrained transformer on the full AG News test split.
The clean metrics provide the baseline before we introduce adversarial-like noise.

In [8]:
clean_rows = []
robustness_rows = []
y_true = test_df['label'].to_numpy()
test_texts = test_df['text'].tolist()

for model_index, (model_name, model) in enumerate(models.items(), start=1):
    print(f'Running {model_name} ({model_index}/{len(models)})...')
    model.fit(train_df['text'], train_df['label'])

    clean_pred = model.predict(test_df['text'])
    clean_metrics = evaluate_predictions(y_true, clean_pred)
    clean_rows.append({'model': model_name, **clean_metrics})

    pd.DataFrame(
        classification_report(
            y_true,
            clean_pred,
            target_names=[LABEL_NAMES[idx] for idx in sorted(LABEL_NAMES)],
            output_dict=True,
            zero_division=0,
        )
    ).T.to_csv(output_dir / f'classification_report_{model_name}_clean.csv')

    save_confusion_matrix(
        y_true,
        clean_pred,
        output_dir / f'confusion_matrix_{model_name}_clean.png',
        title=f'Clean confusion matrix - {model_name}',
    )

clean_df = pd.DataFrame(clean_rows).sort_values(by='macro_f1', ascending=False)
clean_df

Running word_ngram_svm (1/3)...
Running char_ngram_svm (2/3)...
Running transformer_distilbert (3/3)...


,model,accuracy,macro_f1,weighted_f1
2,transformer_distilbert,0.947895,0.947901,0.947901
1,char_ngram_svm,0.927237,0.927144,0.927144
0,word_ngram_svm,0.922237,0.922036,0.922036


## 4. Robustness Benchmark

We now stress test each model with three types of perturbations:
- typo noise,
- word dropout,
- vowel masking.

Each attack is evaluated at three severity levels.

In [9]:
for model_index, (model_name, model) in enumerate(models.items(), start=1):
    for attack_index, (attack_name, severity) in enumerate(ATTACK_PLAN, start=1):
        pert_texts = perturb_corpus(
            test_texts,
            attack_name=attack_name,
            severity=severity,
            seed=seed + model_index * 1000 + attack_index * 100,
        )
        pert_pred = model.predict(pert_texts)
        metrics = evaluate_predictions(y_true, pert_pred)
        robustness_rows.append(
            {
                'model': model_name,
                'attack': attack_name,
                'severity': severity,
                **metrics,
            }
        )

robustness_df = pd.DataFrame(robustness_rows).sort_values(by=['model', 'attack', 'severity'])
robustness_df.head(10)

,model,attack,severity,accuracy,macro_f1,weighted_f1
12,char_ngram_svm,dropout,0.05,0.926974,0.926851,0.926851
13,char_ngram_svm,dropout,0.10,0.921053,0.920936,0.920936
14,char_ngram_svm,dropout,0.20,0.917237,0.917076,0.917076
9,char_ngram_svm,typo,0.05,0.924605,0.924501,0.924501
10,char_ngram_svm,typo,0.10,0.925000,0.924884,0.924884
11,char_ngram_svm,typo,0.20,0.921842,0.921724,0.921724
15,char_ngram_svm,vowel_mask,0.05,0.924868,0.924748,0.924748
16,char_ngram_svm,vowel_mask,0.10,0.924079,0.923947,0.923947
17,char_ngram_svm,vowel_mask,0.20,0.916974,0.916861,0.916861
21,transformer_distilbert,dropout,0.05,0.945395,0.945401,0.945401


In [10]:
save_metric_table(clean_df, output_dir / 'clean_metrics.csv')
save_metric_table(robustness_df, output_dir / 'robustness_metrics.csv')

summary_df = build_model_summary(clean_df, robustness_df)
save_metric_table(summary_df, output_dir / 'model_summary.csv')
save_robustness_curves(robustness_df, output_dir)
save_summary_text(summary_df, output_dir)

summary_df

,model,clean_accuracy,clean_macro_f1,avg_perturbed_accuracy,avg_perturbed_macro_f1,worst_perturbed_macro_f1,mean_macro_f1_drop,worst_macro_f1_drop
1,transformer_distilbert,0.947895,0.947901,0.940629,0.940651,0.930603,0.007250,0.017298
0,char_ngram_svm,0.927237,0.927144,0.922515,0.922392,0.916861,0.004752,0.010283
2,word_ngram_svm,0.922237,0.922036,0.917105,0.916921,0.910790,0.005115,0.011246


## 5. Discussion and Conclusion

This benchmark highlights a common NLP trade-off: strong performance on clean text does not guarantee strong performance under noise.

The word-level model is a strong baseline when the text is intact, while the character-level model should usually degrade more gracefully when spelling or word form is corrupted. The transformer adds a higher-capacity reference point, but it still needs to be judged on robustness as well as clean accuracy. Word dropout is the harshest of the three attacks because it removes content rather than only changing surface form.

The plots and tables saved in `outputs/` will be used in the written report to discuss accuracy, robustness, and the main failure modes of each model.